Importing important libraries

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda
import pandas as pd
import json
import os
import sys

sys.path.append("../src")

from fetch_data import get_data_schema, filter_csv_by_col
from preprocessing import preprocess_data
from call_LLM import remove_redundant_columns, visualize_data
from generate_vis import process_spec, create_vegalite_chart

Loading the dataframe

In [ ]:
# Step 1: Load CSV and preview data
state = {}
state["input_csv"] = "~/Data_viz_agent/data-dashboard/data/apm_endpoint_metrics.csv"
state["output_dir"] = "~/Data_viz_agent/data-dashboard/results/apm_endpoint_metrics"

df = pd.read_csv(state["input_csv"])
state["df"] = df
state["nl_query"] = "Give me an overview of the apm application metrics."

print("🔹 Natural Language Query:")
print(state["nl_query"])
print("\n🔹 Preview of Loaded Data:")
df.head()


Removing redundant columns

In [ ]:
# Step 2: Schema and Redundant Column Removal
state["schema"] = get_data_schema(state["df"])
imp_cols = remove_redundant_columns(state["schema"], state["nl_query"], state["output_dir"])
state["imp_columns"] = imp_cols

print("🔹 Important Columns Identified by LLM:")
print(imp_cols)


LLM Reasoning + Vega Spec generation

In [ ]:
# Step 3: Preprocess + Filter Schema + Generate Vega Spec (LLM output)
state["df"] = preprocess_data(state["df"], state["imp_columns"])
state["filtered_schema"] = filter_csv_by_col(state["schema"], state["imp_columns"])

# Save filtered schema to file
with open(os.path.join(state["output_dir"], "data_schema.json"), "w") as f:
    json.dump(state["filtered_schema"], f)

# LLM Call
spec_list = visualize_data(state["filtered_schema"], state["nl_query"], state["output_dir"])
state["spec_list"] = spec_list

print("🔹 Output from LLM (spec reasoning + raw specs):")
print(json.dumps(spec_list, indent=2))


Displaying the final chart

In [ ]:
# Step 4: Process spec and render chart
final_specs = process_spec(state["spec_list"], state["df"])
state["final_specs"] = final_specs

# Save final spec
with open(os.path.join(state["output_dir"], "spec_list.json"), "w") as f:
    json.dump(final_specs, f)

# Render charts
for index, spec in enumerate(state["final_specs"]):
    create_vegalite_chart(spec, index, state["output_dir"])

print(f"✅ Rendered {len(state['final_specs'])} chart(s). Check the output directory.")
